In [2]:
import cupy as cp

In [3]:
import torch

In [4]:
if torch.cuda.is_available():
    device_properties = torch.cuda.get_device_properties(0)
    print(f"Device Name: {device_properties.name}")
    print(f"Total Memory: {device_properties.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Capability: {device_properties.major}.{device_properties.minor}")
else:
    print("CUDA is not available.")

Device Name: NVIDIA GeForce RTX 4090
Total Memory: 23.53 GB
CUDA Capability: 8.9


In [5]:
from minio import Minio
import os
import pandas as pd
import platform
import time

In [6]:
os.name

'posix'

In [7]:
print(platform.system(), platform.release())

Linux 6.8.0-59-generic


In [2]:
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [3]:
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )

In [6]:
download_data_without_creds('OpenML-har-orig-1.csv')

In [7]:
download_data_without_creds('OpenML-usps-orig-1.csv')

In [11]:
# GaMAC Test

In [8]:
from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

In [9]:
def gamac_test(data, target_measures):
    measures = {"BR": Internal.BR, "OS": Internal.OS, "MCR": Internal.MCR, "SYM": Internal.SYM}
    used_measures = [measures[x] for x in target_measures.split(sep=',')]
    df = pd.read_csv(f'devops/comparison/test-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
        # classes = [int(c[1]) for c in classes]
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    print(f'used measures: {used_measures}')
    result = Gamac(target_measures=tuple(used_measures)).run(table=df, text=None, image=None, classes=classes)
    print(f'optimal.model: {result.model}')
    print(f'clusters: {result.model.labels_}')
    print(f'F1 score: {result.f1_score}')

In [10]:
start = time.time()
gamac_test('OpenML-har-orig-1.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x730bc866b4c0>)>]


/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.2518019676208496s, {'SYM': 1.5896114955220402e-07} ===
=== MEASURES 0.22143244743347168s, {'SYM': 2.0476045442314505e-05} ===
=== ALGO 0.32497072219848633s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 2, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 38.07272720336914s, FailedRun, {'bandwidth': 0.5707802831856472, 'max_iter': 297, 'tol': 6.937995734202628e-05} ===
=== ALGO 3.4948456287384033s, FailedRun, {'name': 'DBSCAN', 'eps': 0.18193782911880624, 'eps_sq': 0.03310137366446394, 'min_samples': 7} ===
=== ALGO 2.8871805667877197s, FailedRun, {'min_cluster_size': 14, 'min_samples': 8} ===
=== MEASURES 0.2316303253173828s, {'SYM': 3.16559062967323e-05} ===
=== ALGO 52.196139097213745s, SuccessRun, {'threshold': 0.6629904421940732, 'branching_factor': 40, 'n_clusters': 6} ===
=== MEASURES 0.22446584701538086s, {'SYM': 0.0003305454017539079} ===
=== ALGO 0.48595237731933594s, SuccessRun, {'name': 'KMeans', 'n_clusters': 11, 'max_iter': 100, 'tol

In [11]:
start = time.time()
gamac_test('OpenML-usps-orig-1.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x730bc866b4c0>)>]


/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.2466731071472168s, {'SYM': 3.1544899428680857e-07} ===
=== MEASURES 0.2359786033630371s, {'SYM': 0.00022743783607118132} ===
=== ALGO 1.641754388809204s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 11, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 37.80021095275879s, FailedRun, {'bandwidth': 0.5707998694640252, 'max_iter': 111, 'tol': 5.073986241377025e-05} ===
=== ALGO 3.109189510345459s, FailedRun, {'name': 'DBSCAN', 'eps': 0.440098510378697, 'eps_sq': 0.19368669883754808, 'min_samples': 8} ===
=== ALGO 17.748647928237915s, FailedRun, {'min_cluster_size': 15, 'min_samples': 15} ===
=== MEASURES 0.2521512508392334s, {'SYM': 0.0001903347839636172} ===
=== ALGO 30.229214191436768s, SuccessRun, {'threshold': 0.6158624688096863, 'branching_factor': 24, 'n_clusters': 15} ===
=== MEASURES 0.2363293170928955s, {'SYM': 0.00030108827036412657} ===
=== ALGO 0.4539363384246826s, SuccessRun, {'name': 'KMeans', 'n_clusters': 12, 'max_iter': 100, 'tol': 